In [ ]:
!pip -q install -r requirements.txt
!apt-get -qq install -y unrar > /dev/null 2>&1 || pip -q install patool rarfile


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, glob, subprocess
EXTRACT_DIR = "/content/xjtu_extracted"
CONDS = ["35Hz12kN", "37.5Hz11kN", "40Hz10kN"]

RAR_DIR = "/content/drive/MyDrive/XJTU-SY_Bearing_Datasets/Data"   # <-- edit if different

def find_condition_root(base):
    for root, dirs, _ in os.walk(base):
        if any(c in dirs for c in CONDS):
            return root
    return None

XJTU_DIR = find_condition_root(EXTRACT_DIR) if os.path.isdir(EXTRACT_DIR) else None
if XJTU_DIR is None:
    first = None
    if os.path.isdir(RAR_DIR):
        cand = sorted(glob.glob(os.path.join(RAR_DIR, "*part01.rar")) +
                      glob.glob(os.path.join(RAR_DIR, "*part1.rar")))
        first = cand[0] if cand else None
    if first is None:  # fallback: search the whole Drive for a part01
        for root, _, files in os.walk("/content/drive/MyDrive"):
            for f in files:
                if f.lower().endswith(("part01.rar", "part1.rar")):
                    first = os.path.join(root, f); break
            if first: break
    assert first, "Could not find *.part01.rar — set RAR_DIR to the folder with the .rar parts."
    print("first part :", first)
    print("all parts  :", sorted(os.path.basename(p) for p in glob.glob(os.path.join(os.path.dirname(first), "*.rar"))))

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    rc = subprocess.run(["unrar", "x", "-o+", first, EXTRACT_DIR + "/"]).returncode
    if rc != 0:
        import patoolib; patoolib.extract_archive(first, outdir=EXTRACT_DIR)
    XJTU_DIR = find_condition_root(EXTRACT_DIR)

assert XJTU_DIR, "Extraction done but no 35Hz12kN/ etc. found under " + EXTRACT_DIR
print("XJTU_DIR   =", XJTU_DIR)
print("conditions :", [c for c in CONDS if os.path.isdir(os.path.join(XJTU_DIR, c))])


In [ ]:
!python cross_paradigm_xjtu.py --data "$XJTU_DIR"


In [ ]:
!python figures.py --xjtu results_xjtu.json
from IPython.display import Image, display
display(Image('figures/fig_xjtu.png'))


In [ ]:
import json
r = json.load(open('results_xjtu.json'))
order = ['Base','+Stats','+HI','+Weibull','+Entropy','+ExpDeg','+LG']
for nm in order:
    if nm in r:
        print(f'{nm:<10} C={r[nm]["c"]:.4f}  AUC={r[nm].get("auc",0):.4f}  IBS={r[nm].get("ibs",0):.4f}')
